# Stage 4 - PCA compression and UMAP visualisation

**Data:** real Porto taxi GPS, 602 H3 cells x 14 standardised features (Stage 3). `synthetic: false`.
**Limitation:** taxis, not buses - no stops, no timetable, no schedule delay.

The rule this notebook enforces:

| Method | Role | Used for clustering? |
|---|---|---|
| PCA | compression: keep the directions that carry variance, drop redundancy | **yes** (Stage 5) |
| UMAP | a 2-D picture for humans | **never** |

Why UMAP must not feed a clusterer: it distorts global distances by design and its axes have no units.
Clusters that appear on a UMAP plot can be artefacts of `n_neighbors`/`min_dist`, not structure in the data.

In [1]:
import json, pathlib, sys, time
sys.path.append(str(pathlib.Path.cwd().parent))

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

from src.io import PROJECT_ROOT
from src import reduce as R

PROC = PROJECT_ROOT / "data" / "processed"
FIG = PROJECT_ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

scaled = pd.read_parquet(PROC / "cells_features_scaled.parquet")
raw = pd.read_parquet(PROC / "cells_features.parquet")
meta = json.loads((PROC / "features_meta.json").read_text())

FEATURE_COLS = meta["feature_cols"]
Z_COLS = ["z_" + c for c in FEATURE_COLS]
print(f"cells: {len(scaled)}  features: {len(FEATURE_COLS)}")
print("features:", FEATURE_COLS)

cells: 602  features: 14
features: ['n_visits', 'n_points', 'n_vehicles', 'dwell_s_per_visit', 'speed_mean', 'speed_p95', 'speed_cv', 'slow_share', 'night_share', 'am_peak_share', 'pm_peak_share', 'weekend_share', 'hour_sin_mean', 'hour_cos_mean']


In [2]:
# 1) Is PCA warranted at all? Look for redundancy: |r| close to 1 between feature pairs.
corr = R.correlation_matrix(scaled, Z_COLS)
corr.index = FEATURE_COLS; corr.columns = FEATURE_COLS

pairs = (corr.where(~np.eye(len(corr), dtype=bool))
             .stack().rename("r").reset_index()
             .rename(columns={"level_0": "a", "level_1": "b"}))
pairs = pairs[pairs["a"] < pairs["b"]].assign(abs_r=lambda d: d["r"].abs())
print("most correlated feature pairs (|r| > 0.7):")
print(pairs[pairs.abs_r > 0.7].sort_values("abs_r", ascending=False).head(10).round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(corr.to_numpy(), cmap="Greys", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURE_COLS))); ax.set_xticklabels(FEATURE_COLS, rotation=70, ha="right", fontsize=8)
ax.set_yticks(range(len(FEATURE_COLS))); ax.set_yticklabels(FEATURE_COLS, fontsize=8)
ax.set_title("Feature correlation (standardised features)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
fig.tight_layout(); fig.savefig(FIG / "stage4_correlation.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage4_correlation.png")

most correlated feature pairs (|r| > 0.7):
                a          b      r  abs_r
       n_vehicles   n_visits  0.971  0.971
         n_points   n_visits  0.967  0.967
       slow_share   speed_cv  0.950  0.950
         n_points n_vehicles  0.942  0.942
dwell_s_per_visit slow_share  0.880  0.880
dwell_s_per_visit   speed_cv  0.844  0.844
       slow_share speed_mean -0.801  0.801
         speed_cv speed_mean -0.795  0.795
       speed_mean  speed_p95  0.786  0.786
dwell_s_per_visit speed_mean -0.768  0.768


saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage4_correlation.png


In [3]:
# 2) Fit PCA with all components so the scree plot can decide how many to keep.
res = R.fit_pca(scaled, Z_COLS, n_components=len(Z_COLS))
evr = res["explained_variance"]
print(evr.round(4).to_string(index=False))

k80 = R.components_for_variance(evr, 0.80)
k90 = R.components_for_variance(evr, 0.90)
print(f"\ncomponents for >=80% variance: {k80}   for >=90%: {k90}")

component  eigenvalue  explained_variance_ratio  cumulative_ratio
      PC1      4.6703                    0.3330            0.3330
      PC2      2.3654                    0.1687            0.5017
      PC3      2.1448                    0.1529            0.6547
      PC4      1.6619                    0.1185            0.7732
      PC5      1.1422                    0.0815            0.8546
      PC6      0.6920                    0.0493            0.9040
      PC7      0.4843                    0.0345            0.9385
      PC8      0.4111                    0.0293            0.9678
      PC9      0.1625                    0.0116            0.9794
     PC10      0.1251                    0.0089            0.9883
     PC11      0.0949                    0.0068            0.9951
     PC12      0.0362                    0.0026            0.9977
     PC13      0.0229                    0.0016            0.9993
     PC14      0.0097                    0.0007            1.0000

component

In [4]:
# 3) Scree + cumulative variance. The elbow matters: after it, extra components add noise.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].bar(evr["component"], evr["explained_variance_ratio"], color="#111111")
axes[0].set_title("scree plot"); axes[0].set_ylabel("variance explained by component")
axes[1].plot(evr["component"], evr["cumulative_ratio"], color="#111111", marker="o", ms=3)
axes[1].axhline(0.8, color="#9A9A9A", ls="--", lw=1); axes[1].axhline(0.9, color="#D9D9D9", ls="--", lw=1)
axes[1].set_title("cumulative variance"); axes[1].set_ylabel("cumulative share")
for ax in axes:
    ax.set_xlabel("principal component"); ax.tick_params(axis="x", rotation=90, labelsize=7)
fig.tight_layout(); fig.savefig(FIG / "stage4_scree.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage4_scree.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage4_scree.png


In [5]:
# 4) What do the components MEAN? A PCA axis is a linear mixture, so it is only
# interpretable through its loadings. Read the sign: + and - pull in opposite directions.
loadings = res["loadings"]
for pc in ["PC1", "PC2", "PC3", "PC4"]:
    print(pc, "->", R.top_loadings(loadings, pc, k=4))

fig, ax = plt.subplots(figsize=(7, 3.6))
im = ax.imshow(loadings[["PC1", "PC2", "PC3", "PC4"]].to_numpy(), cmap="Greys", vmin=-1, vmax=1, aspect="auto")
ax.set_yticks(range(len(FEATURE_COLS))); ax.set_yticklabels(FEATURE_COLS, fontsize=8)
ax.set_xticks(range(4)); ax.set_xticklabels(["PC1", "PC2", "PC3", "PC4"])
ax.set_title("loadings: how each feature contributes to a component")
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
fig.tight_layout(); fig.savefig(FIG / "stage4_loadings.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage4_loadings.png")

PC1 -> [('z_slow_share', 0.4085165882501301), ('z_speed_cv', 0.39954917877641183), ('z_dwell_s_per_visit', 0.39663240925762433), ('z_speed_mean', -0.3934942939087836)]
PC2 -> [('z_n_visits', 0.4899083360516995), ('z_n_vehicles', 0.4890262273686918), ('z_n_points', 0.3940618050762578), ('z_speed_p95', 0.34386576923432205)]
PC3 -> [('z_night_share', 0.5833391683273086), ('z_hour_cos_mean', 0.557026229846327), ('z_weekend_share', 0.46008843894494633), ('z_am_peak_share', -0.2561432155171861)]
PC4 -> [('z_hour_sin_mean', 0.6739570030262709), ('z_am_peak_share', 0.5182787568954008), ('z_pm_peak_share', -0.46202750081637733), ('z_night_share', 0.141107814316203)]


saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage4_loadings.png


In [6]:
# 5) UMAP on the first 10 PCs (or fewer if k80 is smaller). Plot only.
t0 = time.time()
n_pcs_umap = min(k80, 10)
umap_coords, umap_params = R.fit_umap(res["scores"], n_pcs=n_pcs_umap)
print(f"UMAP fitted in {time.time()-t0:.1f}s with {umap_params}")

# Attach a few raw features purely so the picture can be read.
plot_df = umap_coords.merge(raw[["h3_cell", "n_visits", "peak_hour", "night_share"]], on="h3_cell", how="left")
plot_df[["umap_x", "umap_y"]].describe().round(3)

/home/ashira/Documents/Projects/Fleetsense/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/ashira/Documents/Projects/Fleetsense/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP fitted in 17.5s with {'n_pcs': 5, 'n_neighbors': 15, 'min_dist': 0.1, 'metric': 'euclidean', 'random_state': 42}


,umap_x,umap_y
count,602.000,602.000
mean,1.253,6.976
std,2.273,2.251
min,-3.265,2.713
25%,-0.420,5.107
50%,1.118,6.831
75%,2.804,8.795
max,6.388,11.463


In [7]:
# 6) Side-by-side: PCA (2 components) vs UMAP, both coloured by visit count.
pca2 = res["scores"][["h3_cell", "PC1", "PC2"]].merge(
    raw[["h3_cell", "n_visits"]], on="h3_cell", how="left")
col = np.log1p(pca2["n_visits"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
s0 = axes[0].scatter(pca2["PC1"], pca2["PC2"], c=col, cmap="Greys", s=22, linewidths=0.3, edgecolors="#666666")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].set_title(f"PCA: PC1 vs PC2 ({evr['cumulative_ratio'].iloc[1]:.0%} of variance)")
axes[1].scatter(plot_df["umap_x"], plot_df["umap_y"], c=np.log1p(plot_df["n_visits"]),
                cmap="Greys", s=22, linewidths=0.3, edgecolors="#666666")
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")
axes[1].set_title(f"UMAP: {n_pcs_umap} PCs, n_neighbors={umap_params['n_neighbors']}, min_dist={umap_params['min_dist']}")
fig.colorbar(s0, ax=axes, fraction=0.02, pad=0.02, label="log(1 + visits)")
fig.suptitle("colour = visit count (a raw feature, no clustering involved)", y=1.02)
fig.savefig(FIG / "stage4_pca_umap.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage4_pca_umap.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage4_pca_umap.png


In [8]:
# 7) Should you believe the picture? Trustworthiness compares each point's k nearest
# neighbours in the 14-D feature space with its neighbours in the 2-D embedding.
X = scaled[Z_COLS].to_numpy(dtype=float)
trust = R.neighbourhood_trust(
    X,
    {"PCA 2-D": res["scores"][["PC1", "PC2"]].to_numpy(),
     "UMAP 2-D": umap_coords[["umap_x", "umap_y"]].to_numpy()},
    n_neighbors=10,
)
print(trust.to_string(index=False))

embedding  dims  trustworthiness
 UMAP 2-D     2           0.9510
  PCA 2-D     2           0.8417


In [9]:
# 8) Orthogonality check + persist.
score_cols = [c for c in res["scores"].columns if c.startswith("PC")]
off_diag = res["scores"][score_cols].corr().where(~np.eye(len(score_cols), dtype=bool)).abs().max().max()
print("max |corr| between PCA scores (should be ~0):", round(float(off_diag), 6))

res["scores"].to_parquet(PROC / "pca_scores.parquet", index=False)
loadings.to_parquet(PROC / "pca_loadings.parquet")
evr.to_parquet(PROC / "pca_explained_variance.parquet", index=False)
umap_coords.to_parquet(PROC / "umap_coords.parquet", index=False)

out = {
    "stage": 4,
    "data_source": "porto_taxi_uci339",
    "synthetic": False,
    "limitation": meta["limitation"],
    "n_cells": int(len(scaled)),
    "feature_cols": FEATURE_COLS,
    "pca": {"n_components_fitted": int(len(evr)),
            "components_for_80pct": int(k80), "components_for_90pct": int(k90),
            "pc1_ratio": float(evr["explained_variance_ratio"].iloc[0]),
            "two_component_ratio": float(evr["cumulative_ratio"].iloc[1])},
    "umap": umap_params,
    "trustworthiness": trust.to_dict("records"),
    "max_abs_corr_between_pcs": float(off_diag),
}
(PROC / "reduce_meta.json").write_text(json.dumps(out, indent=2, default=str))
print("wrote", PROC / "reduce_meta.json")

max |corr| between PCA scores (should be ~0): 0.0
wrote /home/ashira/Documents/Projects/Fleetsense/data/processed/reduce_meta.json


## Reading the result

* **PCA is compressible only if features are redundant.** Strong pairs (for example `speed_mean` with
  `speed_p95`) mean a handful of components carry most of the variance; the scree elbow tells you where extra
  components start describing noise.
* **Components are mixtures, not variables.** PC1 is typically a "size/volume vs calm" axis. Do not name a
  component after a single feature; read the loadings table and describe the direction of the sign.
* **UMAP's picture is for humans.** Its trustworthiness is reported next to PCA's 2-D projection so you can see
  quantitatively which one preserves neighbourhoods - but the clusterer only ever receives PCA scores (Stage 5).
* Clustering on UMAP output is prohibited in this project: it would let `n_neighbors` and `min_dist` - two
  arbitrary knobs - define the clusters, and no internal metric could detect that.

**What Stage 5 receives:** the PCA score table (and, for DBSCAN, the raw lat/lon with haversine - because
DBSCAN's haversine metric needs coordinates, not z-scores).